In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple

from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.optim as optim

# ---------------------------------------------------------
# 1. Your data: 3D inputs and 1D outputs (maximisation)
# ---------------------------------------------------------
X_raw = np.array([
    [1.71525207e-01, 3.43916870e-01, 2.48737201e-01],
    [2.42114461e-01, 6.44074270e-01, 2.72432809e-01],
    [5.34905720e-01, 3.98500915e-01, 1.73388729e-01],
    [4.92581415e-01, 6.11593188e-01, 3.40176386e-01],
    [1.34621666e-01, 2.19917240e-01, 4.58206220e-01],
    [3.45523271e-01, 9.41359831e-01, 2.69363479e-01],
    [1.51836632e-01, 4.39990619e-01, 9.90881867e-01],
    [6.45502835e-01, 3.97142940e-01, 9.19771338e-01],
    [7.46911945e-01, 2.84196309e-01, 2.26299855e-01],
    [1.70476994e-01, 6.97032401e-01, 1.49169434e-01],
    [2.20549337e-01, 2.97825244e-01, 3.43555344e-01],
    [6.66013659e-01, 6.71985151e-01, 2.46295297e-01],
    [4.68089497e-02, 2.31360241e-01, 7.70617592e-01],
    [6.00097282e-01, 7.25135725e-01, 6.60886415e-02],
    [9.65994849e-01, 8.61119690e-01, 5.66829131e-01],
    [1.06599400e+00, 1.04135900e+00, 1.09088100e+00],
    [4.03482000e-01, 3.82170000e-01, 4.89363000e-01],
    [3.98350000e-01, 1.00000000e-06, 5.43642000e-01],
    [9.62851000e-01, 9.87386000e-01, 4.08750000e-02]
])

y_raw = np.array([
    -0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758,
    -0.11062091, -0.39892551, -0.11386851, -0.13146061, -0.09418956,
    -0.04694741, -0.10596504, -0.11804826, -0.03637783, -0.05675837,
    -0.76942796, -0.03310308, -0.09333459, -0.07627378
])

DEVICE = torch.device("cpu")  # change to "cuda" if you have a GPU

# ---------------------------------------------------------
# 2. PyTorch MLP model
# ---------------------------------------------------------

class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes=(64, 64), dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# ---------------------------------------------------------
# 3a. Deep Ensemble Surrogate (deterministic ensemble)
# ---------------------------------------------------------

@dataclass
class DeepEnsembleSurrogate:
    n_members: int = 5
    hidden_layer_sizes: Tuple[int, ...] = (64, 64)
    dropout: float = 0.0
    n_epochs: int = 2000
    lr: float = 1e-3
    random_state: int = 42

    def __post_init__(self):
        self.models = []
        self.x_scaler = StandardScaler()
        self.y_scaler = StandardScaler()

    def _train_single_model(self, Xs: np.ndarray, ys: np.ndarray, seed: int):
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = MLPRegressorTorch(
            input_dim=Xs.shape[1],
            hidden_sizes=self.hidden_layer_sizes,
            dropout=self.dropout
        ).to(DEVICE)

        optimizer = optim.Adam(model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
        y_tensor = torch.from_numpy(ys.astype(np.float32)).view(-1, 1).to(DEVICE)

        model.train()
        for _ in range(self.n_epochs):
            optimizer.zero_grad()
            preds = model(X_tensor)
            loss = criterion(preds, y_tensor)
            loss.backward()
            optimizer.step()

        return model

    def fit(self, X: np.ndarray, y: np.ndarray):
        # Scale inputs and outputs
        Xs = self.x_scaler.fit_transform(X)
        ys = self.y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

        rng = np.random.RandomState(self.random_state)
        self.models = []

        for i in range(self.n_members):
            seed = int(rng.randint(0, 10_000))
            model = self._train_single_model(Xs, ys, seed)
            self.models.append(model)

    def predict(self, X: np.ndarray, return_std: bool = False):
        Xs = self.x_scaler.transform(X)
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)

        preds_scaled_list = []

        for model in self.models:
            model.eval()
            with torch.no_grad():
                out = model(X_tensor).cpu().numpy().ravel()
            preds_scaled_list.append(out)

        preds_scaled = np.stack(preds_scaled_list, axis=0)  # (n_members, n_samples)

        # Convert ensemble mean/std from scaled space to original y space
        mean_scaled = preds_scaled.mean(axis=0)
        std_scaled = preds_scaled.std(axis=0)

        scale_y = self.y_scaler.scale_[0]
        mean_y = self.y_scaler.mean_[0]

        mean = mean_scaled * scale_y + mean_y
        if not return_std:
            return mean

        std = std_scaled * abs(scale_y)
        return mean, std


# ---------------------------------------------------------
# 3b. MC Dropout surrogate (BNN-style via stochastic dropout)
# ---------------------------------------------------------

@dataclass
class MCDropoutSurrogate:
    hidden_layer_sizes: Tuple[int, ...] = (64, 64)
    dropout: float = 0.1
    n_epochs: int = 2000
    lr: float = 1e-3
    random_state: int = 0
    n_mc_samples: int = 100

    def __post_init__(self):
        self.model = None
        self.x_scaler = StandardScaler()
        self.y_scaler = StandardScaler()

    def fit(self, X: np.ndarray, y: np.ndarray):
        Xs = self.x_scaler.fit_transform(X)
        ys = self.y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        model = MLPRegressorTorch(
            input_dim=Xs.shape[1],
            hidden_sizes=self.hidden_layer_sizes,
            dropout=self.dropout
        ).to(DEVICE)

        optimizer = optim.Adam(model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()

        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
        y_tensor = torch.from_numpy(ys.astype(np.float32)).view(-1, 1).to(DEVICE)

        model.train()
        for _ in range(self.n_epochs):
            optimizer.zero_grad()
            preds = model(X_tensor)
            loss = criterion(preds, y_tensor)
            loss.backward()
            optimizer.step()

        self.model = model

    def predict(self, X: np.ndarray, return_std: bool = False):
        Xs = self.x_scaler.transform(X)
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)

        # Activate dropout at inference for MC sampling
        self.model.train()

        preds_scaled_mc = []
        with torch.no_grad():
            for _ in range(self.n_mc_samples):
                out = self.model(X_tensor).cpu().numpy().ravel()
                preds_scaled_mc.append(out)

        preds_scaled_mc = np.stack(preds_scaled_mc, axis=0)  # (n_mc_samples, n_samples)

        mean_scaled = preds_scaled_mc.mean(axis=0)
        std_scaled = preds_scaled_mc.std(axis=0)

        scale_y = self.y_scaler.scale_[0]
        mean_y = self.y_scaler.mean_[0]

        mean = mean_scaled * scale_y + mean_y

        if not return_std:
            return mean

        std = std_scaled * abs(scale_y)
        return mean, std


# ---------------------------------------------------------
# 4. Acquisition functions: Probability of Improvement & EI
# ---------------------------------------------------------

def acquisition_pi_ei(mu: np.ndarray, sigma: np.ndarray, y_best: float, xi: float = 0.0):
    """
    mu, sigma: predictive mean and std at candidate points
    y_best: current best observed output (maximisation)
    xi: exploration parameter (small positive encourages exploration)
    """
    sigma = np.maximum(sigma, 1e-9)
    gamma = (mu - y_best - xi) / sigma

    pi = norm.cdf(gamma)                  # Probability of Improvement
    ei = (mu - y_best - xi) * pi + sigma * norm.pdf(gamma)  # Expected Improvement
    ei = np.maximum(ei, 0.0)
    return pi, ei


# ---------------------------------------------------------
# 5. Helper: propose next query by maximising EI
# ---------------------------------------------------------

def propose_next_point(surrogate,
                       X_obs: np.ndarray,
                       y_obs: np.ndarray,
                       n_candidates: int = 50000,
                       xi: float = 0.01,
                       random_state: int = 123) -> dict:
    """
    Returns dict with:
    - next_x: suggested next query point (3D)
    - pred_mean, pred_std: surrogate prediction at next_x
    - pi, ei: acquisition values at next_x
    - y_best, x_best: current best observed pair
    - reasoning: text explanation using nearby points & uncertainty
    """
    rng = np.random.RandomState(random_state)

    # Empirical bounds from data, padded slightly
    x_min = X_obs.min(axis=0)
    x_max = X_obs.max(axis=0)
    padding = 0.05 * (x_max - x_min)
    lower = np.maximum(0.0, x_min - padding)
    upper = x_max + padding

    # Sample random candidate points within bounds
    X_cand = rng.uniform(lower, upper, size=(n_candidates, X_obs.shape[1]))

    # Predict mean and std with the surrogate
    mu, sigma = surrogate.predict(X_cand, return_std=True)

    # Current best observation (maximisation problem)
    best_idx = np.argmax(y_obs)
    y_best = y_obs[best_idx]
    x_best = X_obs[best_idx]

    # Acquisition
    pi_all, ei_all = acquisition_pi_ei(mu, sigma, y_best=y_best, xi=xi)

    # Pick candidate with highest EI
    next_idx = np.argmax(ei_all)
    next_x = X_cand[next_idx]
    next_mu = mu[next_idx]
    next_std = sigma[next_idx]
    next_pi = pi_all[next_idx]
    next_ei = ei_all[next_idx]

    # Nearest observed points to the proposed candidate
    dists = np.linalg.norm(X_obs - next_x, axis=1)
    nn_order = np.argsort(dists)[:3]

    reasoning_lines = []
    reasoning_lines.append(
        f"• The proposed point lies within the empirical bounds of past experiments: "
        f"lower={lower.round(3)}, upper={upper.round(3)}."
    )
    reasoning_lines.append(
        f"• The surrogate predicts a mean response of {next_mu:.4f} with uncertainty (std) "
        f"{next_std:.4f} at this point."
    )
    reasoning_lines.append(
        f"• Given the current best observed value y_best={y_best:.4f}, the probability that "
        f"this new point improves on y_best is PI={next_pi:.3f}, and its expected improvement "
        f"EI={next_ei:.5f} is the highest among {n_candidates} sampled candidates."
    )

    reasoning_lines.append("• Nearest previously tested points to the proposed query:")
    for rank, idx in enumerate(nn_order, start=1):
        reasoning_lines.append(
            f"   #{rank}: x={X_obs[idx].round(4)}, y={y_obs[idx]:.4f}, "
            f"distance={dists[idx]:.4f}"
        )

    reasoning_lines.append(
        "• The proposed point balances exploitation (it is near regions with relatively good "
        "observed outputs) and exploration (predictive uncertainty here is still sizeable), "
        "making it a strong candidate to discover a higher local maximum."
    )

    reasoning = "\n".join(reasoning_lines)

    return {
        "next_x": next_x,
        "pred_mean": next_mu,
        "pred_std": next_std,
        "pi": next_pi,
        "ei": next_ei,
        "y_best": y_best,
        "x_best": x_best,
        "reasoning": reasoning
    }


# ---------------------------------------------------------
# 6. Main: choose surrogate, train, propose next query
# ---------------------------------------------------------

def main():
    np.random.seed(0)
    torch.manual_seed(0)

    # ---- Choose which deep-learning surrogate to use ----
    SURROGATE_TYPE = "mc_dropout"   # or "deep_ensemble"

    if SURROGATE_TYPE == "deep_ensemble":
        surrogate = DeepEnsembleSurrogate(
            n_members=5,
            hidden_layer_sizes=(64, 64),
            dropout=0.0,
            n_epochs=2000,
            lr=1e-3,
            random_state=0
        )
    elif SURROGATE_TYPE == "mc_dropout":
        surrogate = MCDropoutSurrogate(
            hidden_layer_sizes=(64, 64),
            dropout=0.1,
            n_epochs=2000,
            lr=1e-3,
            random_state=0,
            n_mc_samples=200
        )
    else:
        raise ValueError(f"Unknown SURROGATE_TYPE: {SURROGATE_TYPE}")

    # Fit surrogate
    surrogate.fit(X_raw, y_raw)

    # Current best observed point
    best_idx = np.argmax(y_raw)
    current_best_x = X_raw[best_idx]
    current_best_y = y_raw[best_idx]

    # Propose next query point via EI
    suggestion = propose_next_point(
        surrogate=surrogate,
        X_obs=X_raw,
        y_obs=y_raw,
        n_candidates=50000,
        xi=0.01,
        random_state=123
    )

    # --------------------------
    # Print results
    # --------------------------
    print("===== CURRENT BEST (LOCAL MAXIMA FROM DATA) =====")
    print(f"Best observed input x_best = {current_best_x}")
    print(f"Best observed output y_best = {current_best_y:.6f}")
    print()

    print(f"===== SUGGESTED NEXT QUERY POINT ({SURROGATE_TYPE}) =====")
    print(f"Next query x_next = {suggestion['next_x']}")
    print(f"Predicted mean f(x_next) = {suggestion['pred_mean']:.6f}")
    print(f"Predictive std at x_next  = {suggestion['pred_std']:.6f}")
    print(f"Probability of improvement PI = {suggestion['pi']:.4f}")
    print(f"Expected improvement EI       = {suggestion['ei']:.6f}")
    print()

    print("===== DETAILED REASONING =====")
    print(suggestion["reasoning"])


if __name__ == "__main__":
    main()


===== CURRENT BEST (LOCAL MAXIMA FROM DATA) =====
Best observed input x_best = [0.403482 0.38217  0.489363]
Best observed output y_best = -0.033103

===== SUGGESTED NEXT QUERY POINT (mc_dropout) =====
Next query x_next = [0.2651598  0.28693127 0.41377776]
Predicted mean f(x_next) = -0.030038
Predictive std at x_next  = 0.011258
Probability of improvement PI = 0.2689
Expected improvement EI       = 0.001850

===== DETAILED REASONING =====
• The proposed point lies within the empirical bounds of past experiments: lower=[0. 0. 0.], upper=[1.117 1.093 1.143].
• The surrogate predicts a mean response of -0.0300 with uncertainty (std) 0.0113 at this point.
• Given the current best observed value y_best=-0.0331, the probability that this new point improves on y_best is PI=0.269, and its expected improvement EI=0.00185 is the highest among 50000 sampled candidates.
• Nearest previously tested points to the proposed query:
   #1: x=[0.2205 0.2978 0.3436], y=-0.0469, distance=0.0839
   #2: x=[0.